# Day 8 — Strict predictor (confidence gate)

Build `predict(image_path)` that retrieves top-k and refuses when similarity is low.

In [ ]:
!pip -q install pandas numpy torch transformers pillow faiss-cpu

In [ ]:
# === Setup ===
from google.colab import drive
drive.mount('/content/drive')

import os, re, time, uuid
import numpy as np
import pandas as pd
from tqdm import tqdm

BASE = "/content/drive/MyDrive/mimic_project"
print("BASE:", BASE)


In [ ]:
import faiss, torch
from PIL import Image
from transformers import CLIPProcessor, CLIPModel

df = pd.read_csv(f"{BASE}/clean_dataset_2696.csv")
fused = np.load(f"{BASE}/fused_embeddings_alpha_0.5.npy").astype("float32")
fused = fused / np.linalg.norm(fused, axis=1, keepdims=True)

index = faiss.IndexFlatIP(fused.shape[1]); index.add(fused)
print("Index size:", index.ntotal)

device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "openai/clip-vit-base-patch32"
model = CLIPModel.from_pretrained(model_name).to(device)
processor = CLIPProcessor.from_pretrained(model_name)
model.eval()
print("Device:", device)

def embed_one_image(image_path: str) -> np.ndarray:
    img = Image.open(image_path).convert("RGB")
    inputs = processor(images=[img], return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        v = model.get_image_features(**inputs)
        v = v / v.norm(dim=-1, keepdim=True)
    return v.cpu().numpy().astype("float32")[0]

def retrieve(v: np.ndarray, k=3):
    s, idx = index.search(v.reshape(1,-1).astype("float32"), k)
    cases=[]
    for j,(i,sc) in enumerate(zip(idx[0].tolist(), s[0].tolist()), start=1):
        cases.append({"case":j,"row_idx":i,"study_id":int(df.loc[i,'study_id']),"score":float(sc),
                      "impression":str(df.loc[i,'impression']).strip()})
    best=float(s[0][0]) if len(s[0]) else 0.0
    return cases, best

def draft_from_cases(cases, k=3):
    bullets=[]
    for c in cases[:k]:
        txt=c["impression"].replace("\n"," ").strip()
        first=re.split(r"(?<=[.!?])\s+", txt)[0].strip()
        if first:
            bullets.append(f"- {first} [Case {c['case']}]")
    return "IMPRESSION:\n" + ("\n".join(bullets) if bullets else "- Unable to draft.")

def predict(image_path: str, k=3, min_score=0.90):
    t0=time.time()
    v=embed_one_image(image_path)
    cases,best=retrieve(v,k=k)
    status="generated" if best>=min_score else "refused"
    draft=draft_from_cases(cases,k=k)
    if status=="refused":
        draft="IMPRESSION:\n- Unable to generate reliable draft due to low retrieval confidence."
    return {"status":status,"confidence":float(best),"latency_ms":int((time.time()-t0)*1000),
            "draft_impression":draft,"retrieved_cases":cases}

# quick test
img = df.loc[0,"image_path"]
out = predict(img,k=3,min_score=0.90)
print(out["status"], round(out["confidence"],3))
print(out["draft_impression"])
